In [6]:
# Cell 4: Load and calculate derived features
import xarray as xr

# Open the downloaded NetCDF file
ds = xr.open_dataset("phy_data.nc")

# Calculate Eddy Kinetic Energy (EKE)
ds['eke'] = 0.5 * (ds['uo']**2 + ds['vo']**2)

# Inspect the updated dataset
print(ds)

<xarray.Dataset> Size: 4GB
Dimensions:    (time: 4018, depth: 1, latitude: 205, longitude: 133)
Coordinates:
  * time       (time) datetime64[ns] 32kB 2015-01-01 2015-01-02 ... 2025-12-31
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 820B 4.0 4.083 4.167 4.25 ... 20.83 20.92 21.0
  * longitude  (longitude) float32 532B 116.0 116.1 116.2 ... 126.8 126.9 127.0
Data variables:
    thetao     (time, depth, latitude, longitude) float64 876MB ...
    zos        (time, latitude, longitude) float64 876MB ...
    uo         (time, depth, latitude, longitude) float64 876MB nan ... 0.02258
    vo         (time, depth, latitude, longitude) float64 876MB nan ... 0.1489
    eke        (time, depth, latitude, longitude) float64 876MB nan ... 0.01135
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:              

In [2]:
import copernicusmarine

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
    variables=["chl"],
    minimum_longitude=116,
    maximum_longitude=127,
    minimum_latitude=4,
    maximum_latitude=21,
    minimum_depth=0.0,         # Changed from 0.49
    maximum_depth=1.0,         # Changed from 0.5 to catch the 0.50576m layer
    start_datetime="2015-01-01",
    end_datetime="2025-12-31",
    output_filename="bgc_data.nc"
)

/Users/gianne/Development/Python/Parola/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO - 2026-07-09T00:54:26Z - Selected dataset version: "202406"
INFO - 2026-07-09T00:54:26Z - Selected dataset part: "default"
WARNING - 2026-07-09T00:54:26Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.5057600140571594, 5902.0576171875]


KeyboardInterrupt: 

In [11]:
import copernicusmarine

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc_my_0.25deg_P1D-m",
    variables=["chl"],
    minimum_longitude=116,
    maximum_longitude=131,
    minimum_latitude=4,
    maximum_latitude=21,
    minimum_depth=0.0,         # Changed from 0.49
    maximum_depth=1.0,         # Changed from 0.5 to catch the 0.50576m layer
    start_datetime="2015-01-01",
    end_datetime="2025-12-31",
    output_filename="bgc_data.nc"
)

INFO - 2026-07-09T03:47:57Z - Selected dataset version: "202406"
INFO - 2026-07-09T03:47:57Z - Selected dataset part: "default"
WARNING - 2026-07-09T03:47:57Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.5057600140571594, 5902.0576171875]
100%|██████████| [00:09<00:00]
INFO - 2026-07-09T03:48:10Z - Total size of the download: 64.56 MB.


ResponseSubset(file_path=PosixPath('bgc_data_(1).nc'), output_directory=PosixPath('.'), filename='bgc_data_(1).nc', file_size=64.56170992366413, data_transfer_size=256.8989312977099, variables=['chl'], coordinates_extent=[GeographicalExtent(minimum=116.0, maximum=131.0, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=4.0, maximum=21.0, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='2015-01-01T00:00:00+00:00', maximum='2025-12-31T00:00:00+00:00', unit='iso8601', coordinate_id='time'), GeographicalExtent(minimum=0.5057600140571594, maximum=0.5057600140571594, unit='m', coordinate_id='depth')], status='000', message='The request was successful.', file_status='DOWNLOADED', file_names=None)

In [15]:
import xarray as xr

phy = xr.open_dataset("phy_data_v2.nc")
bgc = xr.open_dataset("bgc_data.nc")

chl_surface = bgc.chl.sel(depth=bgc.depth.values[0], method="nearest") if "depth" in bgc.dims else bgc.chl

# Interpolate chl (0.25deg) onto phy's grid (0.083deg) — bilinear via xarray's interp
chl_regridded = chl_surface.interp(
    latitude=phy.latitude,
    longitude=phy.longitude,
    method="linear",
)

# Merge into one master environmental dataset
env = phy.copy()
env["chl"] = chl_regridded
env.to_netcdf("env_master.nc")
print(env)

<xarray.Dataset> Size: 7GB
Dimensions:    (time: 4018, depth: 1, latitude: 253, longitude: 181)
Coordinates:
  * time       (time) datetime64[ns] 32kB 2015-01-01 2015-01-02 ... 2025-12-31
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 1kB 2.0 2.083 2.167 2.25 ... 22.83 22.92 23.0
  * longitude  (longitude) float32 724B 116.0 116.1 116.2 ... 130.8 130.9 131.0
Data variables:
    thetao     (time, depth, latitude, longitude) float64 1GB ...
    zos        (time, latitude, longitude) float64 1GB ...
    uo         (time, depth, latitude, longitude) float64 1GB ...
    vo         (time, depth, latitude, longitude) float64 1GB ...
    chl        (time, latitude, longitude) float32 736MB nan nan nan ... nan nan
Attributes: (12/25)
    Conventions:               CF-1.4
    bulletin_date:             2021-07-07 00:00:00
    bulletin_type:             operational
    comment:                   CMEMS product
    domain_name:               GL12
    easting:             

In [16]:
import polars as pl
import xarray as xr
import numpy as np

vbd = pl.scan_parquet("vbd_qc_filtered_ph_core.parquet").collect()
env = xr.open_dataset("env_master.nc")

lat_grid = env.latitude.values
lon_grid = env.longitude.values

def nearest_idx(vals, grid):
    return np.searchsorted(grid, vals) - (
        np.abs(grid[np.clip(np.searchsorted(grid, vals), 0, len(grid)-1)] - vals)
        < np.abs(grid[np.clip(np.searchsorted(grid, vals)-1, 0, len(grid)-1)] - vals)
    ).astype(int)

vbd_pd = vbd.to_pandas()
vbd_pd["lat_idx"] = np.searchsorted(lat_grid, vbd_pd["Lat_DNB"])
vbd_pd["lon_idx"] = np.searchsorted(lon_grid, vbd_pd["Lon_DNB"])
vbd_pd["grid_lat"] = lat_grid[vbd_pd["lat_idx"].clip(0, len(lat_grid)-1)]
vbd_pd["grid_lon"] = lon_grid[vbd_pd["lon_idx"].clip(0, len(lon_grid)-1)]

daily_presence = (
    vbd_pd.groupby(["detection_date", "grid_lat", "grid_lon"])
    .size()
    .reset_index(name="boat_count")
)
daily_presence["presence"] = 1
daily_presence.to_parquet("vbd_gridded_presence.parquet")
print(daily_presence.shape)

(765441, 5)


In [17]:
import pandas as pd
import numpy as np

presence = pd.read_parquet("vbd_gridded_presence.parquet")
all_dates = pd.date_range("2015-01-01", "2025-12-31", freq="D")
all_cells = presence[["grid_lat", "grid_lon"]].drop_duplicates()

rng = np.random.default_rng(42)
n_absences = len(presence) * 2  # 2:1 negative:positive ratio, adjust as needed

sampled_dates = rng.choice(all_dates, size=n_absences)
sampled_cells = all_cells.sample(n=n_absences, replace=True, random_state=42).reset_index(drop=True)

pseudo_absence = pd.DataFrame({
    "detection_date": sampled_dates,
    "grid_lat": sampled_cells["grid_lat"],
    "grid_lon": sampled_cells["grid_lon"],
})

# Drop any that accidentally match a real presence record (same date+cell)
merged_check = pseudo_absence.merge(
    presence[["detection_date", "grid_lat", "grid_lon"]],
    on=["detection_date", "grid_lat", "grid_lon"],
    how="left", indicator=True
)
pseudo_absence_clean = merged_check[merged_check["_merge"] == "left_only"].drop(columns="_merge")
pseudo_absence_clean["boat_count"] = 0
pseudo_absence_clean["presence"] = 0

pseudo_absence_clean.to_parquet("vbd_pseudo_absence.parquet")
print(pseudo_absence_clean.shape)

(1514838, 5)


In [19]:
import xarray as xr
import numpy as np

env = xr.open_dataset("env_master.nc")

# Drop the leftover size-1 depth dimension from all variables
env = env.squeeze("depth", drop=True)

print(env.thetao.dims, env.thetao.shape)  # should now be (time, latitude, longitude)

# Frontal gradients
sst_grad_lat, sst_grad_lon = np.gradient(env.thetao.values, axis=(-2, -1))
env["sst_frontal_gradient"] = (("time", "latitude", "longitude"),
                                 np.sqrt(sst_grad_lat**2 + sst_grad_lon**2))

chl_grad_lat, chl_grad_lon = np.gradient(env.chl.values, axis=(-2, -1))
env["chl_frontal_gradient"] = (("time", "latitude", "longitude"),
                                 np.sqrt(chl_grad_lat**2 + chl_grad_lon**2))

# Monthly climatology + anomaly
clim = env.groupby("time.month").mean(dim="time")
sst_anomaly = env.thetao.groupby("time.month") - clim.thetao
env["sst_anomaly"] = sst_anomaly

env.to_netcdf("env_features.nc")

('time', 'latitude', 'longitude') (4018, 253, 181)


In [4]:
import pandas as pd
import xarray as xr

presence = pd.read_parquet("vbd_gridded_presence.parquet")
absence = pd.read_parquet("vbd_pseudo_absence.parquet")
labels = pd.concat([presence, absence], ignore_index=True)

env = xr.open_dataset("env_features_filled_v3.nc")

# Build DataArray indexers from your label points — this triggers xarray's
# vectorized ("advanced") indexing: it pulls exactly len(labels) points,
# not the full time*lat*lon cross product.
time_idx = xr.DataArray(labels["detection_date"].values, dims="points")
lat_idx = xr.DataArray(labels["grid_lat"].values, dims="points")
lon_idx = xr.DataArray(labels["grid_lon"].values, dims="points")

sampled = env[["thetao", "zos", "uo", "vo", "chl",
               "sst_frontal_gradient", "chl_frontal_gradient", "sst_anomaly"]].sel(
    time=time_idx, latitude=lat_idx, longitude=lon_idx, method="nearest"
)

sampled_df = sampled.to_dataframe().reset_index()
final = pd.concat([labels.reset_index(drop=True), sampled_df.drop(columns=["points"], errors="ignore")], axis=1)

final.to_csv("lightgbm_ready_dataset_v5.csv", index=False)
print(final.shape, final.isna().sum())

(2280279, 17) detection_date               0
grid_lat                     0
grid_lon                     0
boat_count                   0
presence                     0
thetao                    4499
zos                       4499
uo                        4499
vo                        4499
chl                     137623
sst_frontal_gradient     39900
chl_frontal_gradient    203200
sst_anomaly               4499
latitude                     0
longitude                    0
time                         0
month                        0
dtype: int64


In [3]:
import xarray as xr

env = xr.open_dataset("env_features.nc")

env_filled = env.interpolate_na(dim="longitude", method="nearest", limit=2)
env_filled = env_filled.interpolate_na(dim="latitude", method="nearest", limit=2)

env_filled.to_netcdf("env_features_filled.nc")

In [4]:
import pandas as pd
import xarray as xr

presence = pd.read_parquet("vbd_gridded_presence.parquet")
absence = pd.read_parquet("vbd_pseudo_absence.parquet")
labels = pd.concat([presence, absence], ignore_index=True)

env = xr.open_dataset("env_features_filled.nc")

time_idx = xr.DataArray(labels["detection_date"].values, dims="points")
lat_idx = xr.DataArray(labels["grid_lat"].values, dims="points")
lon_idx = xr.DataArray(labels["grid_lon"].values, dims="points")

sampled = env[["thetao", "zos", "uo", "vo", "chl",
               "sst_frontal_gradient", "chl_frontal_gradient", "sst_anomaly"]].sel(
    time=time_idx, latitude=lat_idx, longitude=lon_idx, method="nearest"
)

sampled_df = sampled.to_dataframe().reset_index()
final = pd.concat([labels.reset_index(drop=True), sampled_df.drop(columns=["points"], errors="ignore")], axis=1)

final.to_csv("lightgbm_ready_dataset_v3.csv", index=False)
print(final.shape, final.isna().sum())

(2280279, 17) detection_date               0
grid_lat                     0
grid_lon                     0
boat_count                   0
presence                     0
thetao                    4499
zos                       4499
uo                        4499
vo                        4499
chl                     332954
sst_frontal_gradient     79186
chl_frontal_gradient    514870
sst_anomaly               4499
latitude                     0
longitude                    0
time                         0
month                        0
dtype: int64


In [ ]:
import xarray as xr

env = xr.open_dataset("env_features.nc")

# thetao/zos/uo/vo already handled well with limit=2 — leave as is
# chl needs a wider reach since it's regridded from a 3x coarser source
env["chl"] = env.chl.interpolate_na(dim="longitude", method="nearest", limit=6)
env["chl"] = env.chl.interpolate_na(dim="latitude", method="nearest", limit=6)

env["thetao"] = env.thetao.interpolate_na(dim="longitude", method="nearest", limit=2)
env["thetao"] = env.thetao.interpolate_na(dim="latitude", method="nearest", limit=2)
env["zos"] = env.zos.interpolate_na(dim="longitude", method="nearest", limit=2)
env["zos"] = env.zos.interpolate_na(dim="latitude", method="nearest", limit=2)
env["uo"] = env.uo.interpolate_na(dim="longitude", method="nearest", limit=2)
env["uo"] = env.uo.interpolate_na(dim="latitude", method="nearest", limit=2)
env["vo"] = env.vo.interpolate_na(dim="longitude", method="nearest", limit=2)
env["vo"] = env.vo.interpolate_na(dim="latitude", method="nearest", limit=2)

# Recompute gradients AFTER filling, so gradients aren't still poisoned by
# the NaNs that used to be there
import numpy as np
sst_grad_lat, sst_grad_lon = np.gradient(env.thetao.values, axis=(-2, -1))
env["sst_frontal_gradient"] = (("time","latitude","longitude"), np.sqrt(sst_grad_lat**2+sst_grad_lon**2))
chl_grad_lat, chl_grad_lon = np.gradient(env.chl.values, axis=(-2, -1))
env["chl_frontal_gradient"] = (("time","latitude","longitude"), np.sqrt(chl_grad_lat**2+chl_grad_lon**2))

env.to_netcdf("env_features_filled_v2.nc")

In [3]:
import xarray as xr

env = xr.open_dataset("env_features_filled_v2.nc")

# Only this was stale — recompute from the already-filled thetao
clim = env.thetao.groupby("time.month").mean(dim="time")
env["sst_anomaly"] = env.thetao.groupby("time.month") - clim

env.to_netcdf("env_features_filled_v3.nc")

In [14]:
pip install scipy

Python(14420) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 14.2 MB/s  0:00:01m0:00:0100:01

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import copernicusmarine

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "zos", "uo", "vo"],
    minimum_longitude=116.0,
    maximum_longitude=131.0,
    minimum_latitude=2.0,
    maximum_latitude=23.0,
    start_datetime="2015-01-01T00:00:00",
    end_datetime="2025-12-31T23:59:59",
    minimum_depth=0.49402499198913574,
    maximum_depth=0.49402499198913574,
    output_filename="phy_data_v2.nc",
    output_directory="./",
)

import copernicusmarine

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc-pft_my_0.25deg_P1D-m",
    variables=["chl"],
    minimum_longitude=116.0,
    maximum_longitude=131.0,
    minimum_latitude=2.0,
    maximum_latitude=23.0,
    start_datetime="2015-01-01T00:00:00",
    end_datetime="2025-12-31T23:59:59",
    minimum_depth=0.49402499198913574,
    maximum_depth=0.49402499198913574,
    output_filename="bgc_data.nc",
    output_directory="./",
)

INFO - 2026-07-09T03:45:38Z - Selected dataset version: "202311"
INFO - 2026-07-09T03:45:38Z - Selected dataset part: "default"
 85%|████████▍ | [01:11<00:12]


KeyboardInterrupt: 

In [1]:

# Cell 1: Install required libraries
!pip install copernicusmarine xarray netcdf4 dask

# Cell 2: Authenticate (Enter credentials when prompted, or pass via flags)
!copernicusmarine login



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 9.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 9.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 11.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 11.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 12.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.5 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 13.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 8.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61/61 [copernicusmarine][dask]c-ext-p

In [4]:
import copernicusmarine
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime

today = datetime.now().strftime('%Y-%m-%d')
current_month = datetime.now().month

print(f"Fetching live data for {today}...")

# 1. Fetch Temperature (thetao)
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m",
    variables=["thetao"],
    minimum_longitude=116.0, maximum_longitude=131.0,
    minimum_latitude=2.0, maximum_latitude=23.0,
    start_datetime=f"{today}T00:00:00", end_datetime=f"{today}T23:59:59",
    minimum_depth=0.0, maximum_depth=1.0, # FIXED DEPTH BOUNDS
    output_filename="live_thetao.nc", overwrite=True
)

# 2. Fetch Currents (uo, vo)
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m",
    variables=["uo", "vo"],
    minimum_longitude=116.0, maximum_longitude=131.0,
    minimum_latitude=2.0, maximum_latitude=23.0,
    start_datetime=f"{today}T00:00:00", end_datetime=f"{today}T23:59:59",
    minimum_depth=0.0, maximum_depth=1.0, # FIXED DEPTH BOUNDS
    output_filename="live_cur.nc", overwrite=True
)

# 3. Fetch Sea Level (zos) - 2D variable, no depth dimension
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_anfc_0.083deg_P1D-m",
    variables=["zos"],
    minimum_longitude=116.0, maximum_longitude=131.0,
    minimum_latitude=2.0, maximum_latitude=23.0,
    start_datetime=f"{today}T00:00:00", end_datetime=f"{today}T23:59:59",
    output_filename="live_zos.nc", overwrite=True
)

# 4. Fetch Chlorophyll (chl)
copernicusmarine.subset(
    dataset_id="cmems_mod_glo_bgc-pft_anfc_0.25deg_P1D-m",
    variables=["chl"],
    minimum_longitude=116.0, maximum_longitude=131.0,
    minimum_latitude=2.0, maximum_latitude=23.0,
    start_datetime=f"{today}T00:00:00", end_datetime=f"{today}T23:59:59",
    minimum_depth=0.0, maximum_depth=1.0, # FIXED DEPTH BOUNDS
    output_filename="live_bgc.nc", overwrite=True
)

print("Merging and regridding...")
# Squeeze depth to match coordinates natively
ds_thetao = xr.open_dataset("live_thetao.nc").squeeze("depth", drop=True)
ds_cur = xr.open_dataset("live_cur.nc").squeeze("depth", drop=True)
ds_zos = xr.open_dataset("live_zos.nc")
bgc = xr.open_dataset("live_bgc.nc").squeeze("depth", drop=True)

# Merge physics datasets seamlessly
phy = xr.merge([ds_thetao, ds_cur, ds_zos])

ocean_mask = phy["thetao"].notnull()
phy["is_ocean"] = ocean_mask

# Regrid CHL (0.25) to PHY grid (0.083)
chl_regridded = bgc.chl.interp(
    latitude=phy.latitude, longitude=phy.longitude, method="linear"
)
phy["chl"] = chl_regridded

# Fill edge NaNs to prevent gradient calculation errors
for var in ["thetao", "zos", "uo", "vo", "chl"]:
    phy[var] = phy[var].interpolate_na(dim="longitude", method="nearest", limit=6)
    phy[var] = phy[var].interpolate_na(dim="latitude", method="nearest", limit=6)

print("Calculating derivations...")
sst_grad_lat, sst_grad_lon = np.gradient(phy.thetao.values, axis=(-2, -1))
phy["sst_frontal_gradient"] = (("time", "latitude", "longitude"), np.sqrt(sst_grad_lat**2 + sst_grad_lon**2))

chl_grad_lat, chl_grad_lon = np.gradient(phy.chl.values, axis=(-2, -1))
phy["chl_frontal_gradient"] = (("time", "latitude", "longitude"), np.sqrt(chl_grad_lat**2 + chl_grad_lon**2))

# Use your actual historical master file to calculate the current month's SST anomaly
historical_env = xr.open_dataset("env_features_filled_v3.nc")
monthly_clim = historical_env.thetao.sel(time=historical_env['time.month'] == current_month).mean(dim="time")
phy["sst_anomaly"] = phy.thetao - monthly_clim

print("Flattening for inference...")
df_today = phy.to_dataframe().reset_index()

df_today = df_today[df_today["is_ocean"] == True].copy()

# Format output to match your LightGBM expectations
df_today.rename(columns={"latitude": "grid_lat", "longitude": "grid_lon"}, inplace=True)
df_today['detection_date'] = today
df_today['month'] = current_month

# Export exactly the columns your model expects
output_cols = [
    'detection_date', 'grid_lat', 'grid_lon', 
    'thetao', 'zos', 'uo', 'vo', 'chl', 
    'sst_frontal_gradient', 'chl_frontal_gradient', 'sst_anomaly', 'month'
]
df_today[output_cols].to_csv("copernicus_live_today.csv", index=False)
print("Saved to copernicus_live_today.csv!")

Fetching live data for 2026-07-10...


INFO - 2026-07-10T07:00:20Z - Selected dataset version: "202406"
INFO - 2026-07-10T07:00:20Z - Selected dataset part: "default"
WARNING - 2026-07-10T07:00:20Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-07-10T07:00:25Z - Total size of the download: 192.29 KB.
INFO - 2026-07-10T07:00:30Z - Selected dataset version: "202406"
INFO - 2026-07-10T07:00:30Z - Selected dataset part: "default"
WARNING - 2026-07-10T07:00:30Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-07-10T07:00:35Z - Total size of the download: 371.27 KB.
INFO - 2026-07-10T07:00:40Z - Selected dataset version: "202406"
INFO - 2026-07-10T07:00:40Z - Selected dataset part: "default"
INFO - 2026-07-10T07:00:45Z - Total size of the download: 192.29 KB.
INFO - 2026-07-10T07:00:50Z - Selected dataset version: "202311"
INFO - 2

Merging and regridding...
Calculating derivations...
Flattening for inference...
Saved to copernicus_live_today.csv!
